# 02 — Maturity Scaling

Non-securitisation gross JTDs are scaled by a maturity weight before netting:

```
weight = 1.0                         if maturity >= 1.0 Y
weight = 0.25                        if maturity < 0.25 Y  (floor)
weight = (maturity - 0.25) / 0.75   if 0.25 ≤ maturity < 1.0 Y  (linear ramp)

scaled_jtd = gross_jtd * weight
```

*Regulatory refs*: Basel MAR22.12; US NPR § 210(a)(2)(iii)

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Maturity weight function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

maturities = np.linspace(0, 2.5, 500)

def weight(m: float) -> float:
    if m < 0.25:
        return 0.25
    if m < 1.0:
        return (m - 0.25) / 0.75
    return 1.0

weights = [weight(m) for m in maturities]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(maturities, weights, lw=2, color="#2563EB")
ax.axhline(0.25, ls="--", color="#94A3B8", lw=1, label="Floor (0.25)")
ax.axhline(1.0,  ls="--", color="#64748B", lw=1, label="Full weight (1.0)")
ax.axvline(0.25, ls=":", color="#CBD5E1", lw=1)
ax.axvline(1.0,  ls=":", color="#CBD5E1", lw=1)
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Maturity weight")
ax.set_title("Non-securitisation maturity weight (US NPR § 210(a)(2)(iii))")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## beta-tech maturity ladder

The four `beta-tech` LONG SENIOR\_DEBT positions span 0.1 Y through 5.0 Y, demonstrating:

| Maturity | Regime | Weight |
|----------|--------|--------|
| 0.1 Y | Floor | 0.25 |
| 0.5 Y | Linear | 0.50 |
| 1.0 Y | Full weight | 1.00 |
| 5.0 Y | Full weight | 1.00 |

In [ ]:
from frtb_drc.gross_jtd import calculate_gross_jtd
from frtb_drc.maturity import scale_gross_jtd
from frtb_drc.reference_data import US_NPR_2_0_PROFILE_ID

beta_positions = [p for p in positions if "beta" in p.position_id]
rows = []
for p in sorted(beta_positions, key=lambda x: x.maturity_years):
    gross = calculate_gross_jtd(p, profile_id=US_NPR_2_0_PROFILE_ID)
    scaled = scale_gross_jtd(gross, p.maturity_years, profile_id=US_NPR_2_0_PROFILE_ID)
    rows.append([
        f"{p.maturity_years:.1f} Y",
        "floor" if scaled.floor_applied else ("full" if p.maturity_years >= 1.0 else "linear"),
        f"{scaled.maturity_weight:.4f}",
        f"{gross.gross_jtd:>12,.2f}",
        f"{scaled.scaled_jtd:>12,.2f}",
    ])

display(markdown_table(
    ["Maturity", "Regime", "Weight", "Gross JTD", "Scaled JTD"], rows
))

## All scaled JTDs

In [ ]:
from frtb_drc.gross_jtd import calculate_gross_jtd
from frtb_drc.maturity import scale_gross_jtd

gross_results = [calculate_gross_jtd(p, profile_id=US_NPR_2_0_PROFILE_ID) for p in positions]
scaled_results = [
    scale_gross_jtd(g, p.maturity_years, profile_id=US_NPR_2_0_PROFILE_ID)
    for g, p in zip(gross_results, positions)
]

rows = [
    [
        s.position_id,
        p.bucket_key,
        f"{p.maturity_years:.2f}",
        f"{s.maturity_weight:.4f}",
        "Y" if s.floor_applied else "",
        f"{s.scaled_jtd:>12,.2f}",
    ]
    for s, p in zip(scaled_results, positions)
    if s.scaled_jtd > 0
]
display(markdown_table(
    ["Position", "Bucket", "Maturity (Y)", "Weight", "Floor?", "Scaled JTD"], rows
))

## Scaled JTD vs maturity scatter

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
for bucket, colour in [
    ("CORPORATE", "#2563EB"), ("NON_US_SOVEREIGN", "#059669"),
    ("PSE_GSE", "#D97706"), ("DEFAULTED", "#DC2626"),
]:
    xs = [p.maturity_years for p, s in zip(positions, scaled_results)
          if p.bucket_key == bucket and s.scaled_jtd > 0]
    ys = [s.scaled_jtd / 1e6 for p, s in zip(positions, scaled_results)
          if p.bucket_key == bucket and s.scaled_jtd > 0]
    ax.scatter(xs, ys, label=bucket, alpha=0.7, color=colour, s=60)

mats = np.linspace(0, 2.5, 200)
ws = [weight(m) for m in mats]
ax.set_xlabel("Maturity (years)")
ax.set_ylabel("Scaled JTD (USD M)")
ax.set_title("Scaled JTD vs maturity")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()